# 06 ? Analytical Dataset Design

## Objective
Design two analytical products from the validated EM2023 relational layer: a trip-level dataset for travel and mode-choice analysis, followed by an origin-destination (OD) matrix by ZAT. This notebook builds only the first, unweighted trip-level dataset.

## Product 1: trip-level dataset
- **Unit of observation:** one row per record in `viajes` (one surveyed trip record).
- **Input:** `data/relational/em2023_relational.sqlite`, table `viajes`.
- **Fields:** retain every source field and exact header from `viajes`, including identifiers, origin/destination, time and duration, mode, purpose, reported stage count, person/household attributes present in this table, and `fexp_vj`.
- **Joins and filtering:** none. The trip table already carries the selected fields; avoiding joins prevents accidental row multiplication. The empirically validated trip key is checked for uniqueness and nulls.
- **Transformations:** none. Values remain as stored text. The expansion factor is retained but not applied; mode and geography codes are not recoded.
- **Output:** local CSV at `data/analytical/em2023_trip_dataset.csv`; field-level lineage and provisional role labels are catalogued in `docs/data_dictionary/analytical_trip_dataset_fields.csv`.

## Product 2: OD matrix by ZAT (design only)
- **Target unit:** an origin-ZAT ? destination-ZAT pair, optionally stratified by a documented time period and mode definition.
- **Candidate inputs:** `viajes.zat_ori`, `viajes.zat_des`, and `viajes.fexp_vj`; the unweighted trip dataset can support exploratory record counts only.
- **Not yet produced:** a weighted OD matrix. Before aggregation, verify ZAT field definitions and valid geography, the survey universe and meaning of `fexp_vj`, treatment of missing/invalid zones, time-period rules, and the EM2023-specific methodology. The Stage V contract methodology is contextual evidence and is not assumed to apply automatically.

## Limitations
The first dataset is a source-preserving analytical starting point, not a finalized choice-model sample. Eligibility, trip purpose, mode grouping, missing-value handling, survey design, and weight use require documented research and survey-method decisions. Source headers retain whitespace and original language by design.


In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src' / 'mode_choice').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the project root from the notebook path.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print(f'Project root: {PROJECT_ROOT}')


Project root: D:\JAIME_ALEJANDRO_GARCIA\01_ACADEMIC\2026_fall\508_Transportation_planning\Transportatio_Planning\508_TP_Proyect


In [2]:
from mode_choice.trip_dataset import build_trip_dataset

dataset_path, field_catalog_path, summary = build_trip_dataset(PROJECT_ROOT)
print('Trip-level dataset build completed.')
print(f"Dataset: {dataset_path.relative_to(PROJECT_ROOT)}")
print(f"Field catalog: {field_catalog_path.relative_to(PROJECT_ROOT)}")
print(summary)


Trip-level dataset build completed.
Dataset: data\analytical\em2023_trip_dataset.csv
Field catalog: docs\data_dictionary\analytical_trip_dataset_fields.csv
{'source_trip_rows': 100174, 'dataset_rows': 100174, 'source_columns': 47, 'dataset_columns': 47, 'distinct_trip_keys': 100174, 'null_trip_keys': 0}


## Acceptance checks
The build stops before publishing a new dataset if the database is missing, the trip key is not unique/non-null, a required source field is missing, or the staged CSV does not match the SQLite header and row counts. The process reads the database in read-only mode and does not access or change `data/sources/raw/`.

Review the field catalog before selecting a final modeling sample. The next analytical step is to document and validate expansion and ZAT rules, then build a separately specified OD product.
